## Two-year Encoded VaR draft

Short-sample run of the same VAE idea. Input is `data/Nifty50_2yr.csv` (from 27 September 2021). The full-sample study, and the AE ratio of 1.04, is `Encoded_VaR_FCP.ipynb`.


In [1]:
# Earlier Encoded VaR experiment on a two-year Nifty extract.
# Original Colab file: /content/2y_Nifty50_data.xlsx.
# This repository's matching file is data/Nifty50_2yr.csv
# (Dates, NIFTY Index, 525 rows from 27 September 2021).
# Window 90, step 21, test block 10, 95% VaR, 30 epochs.
# See Encoded VaR.md. The reported AE ratio of 1.04 is from Encoded_VaR_FCP.ipynb,
# which uses the full 2005-2023 workbook, not this short sample.

# Shared dataset locator.
# The workbook and the two-year CSV live in the project data/ folder.
# Running this notebook from a method folder (Historical, GARCH, ...) or from
# the project root both resolve. Machine-specific paths from the original
# Colab / local drives are no longer required.
from pathlib import Path

def find_nifty_file(filename="Nifty50_data.xlsx"):
    """Return the first existing copy of a file under data/ or the working directory."""
    here = Path.cwd().resolve()
    candidates = [
        here / "data" / filename,
        here.parent / "data" / filename,
        here / filename,
    ]
    for path in candidates:
        if path.exists():
            return str(path.resolve())
    raise FileNotFoundError(
        f"Could not find {filename}. Open the notebook from the project folder "
        "or from its method folder so that data/ is visible."
    )

NIFTY_PATH = find_nifty_file("Nifty50_data.xlsx")
print("Using dataset:", NIFTY_PATH)

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

class VariationalAutoencoder:
    def __init__(self, input_dim=90, latent_dim=32, beta=1.0):
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.beta = beta
        self.build_vae()

    def build_vae(self):
        inputs = Input(shape=(self.input_dim,))
        x = Dense(128, activation='relu')(inputs)
        x = Dense(64, activation='relu')(x)
        z_mean = Dense(self.latent_dim)(x)
        z_log_var = Dense(self.latent_dim)(x)

        def sampling(args):
            z_mean, z_log_var = args
            epsilon = K.random_normal(shape=(K.shape(z_mean)[0], self.latent_dim), mean=0., stddev=1.0)
            return z_mean + K.exp(0.5 * z_log_var) * epsilon

        z = Lambda(sampling)([z_mean, z_log_var])
        latent_inputs = Input(shape=(self.latent_dim,))
        x = Dense(64, activation='relu')(latent_inputs)
        x = Dense(128, activation='relu')(x)
        decoder_outputs = Dense(self.input_dim)(x)
        decoder = Model(latent_inputs, decoder_outputs, name="decoder")

        class VAE(keras.Model):
            def __init__(self, encoder, decoder, beta, **kwargs):
                super().__init__(**kwargs)
                self.encoder = encoder
                self.decoder = decoder
                self.beta = beta

            def train_step(self, data):
                with tf.GradientTape() as tape:
                    z_mean, z_log_var, z = self.encoder(data)
                    reconstruction = self.decoder(z)
                    reconstruction_loss = tf.reduce_mean(keras.losses.mse(data, reconstruction))
                    kl_loss = -0.5 * tf.reduce_mean(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
                    total_loss = reconstruction_loss + self.beta * kl_loss

                grads = tape.gradient(total_loss, self.trainable_weights)
                self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
                return {"loss": total_loss, "reconstruction_loss": reconstruction_loss, "kl_loss": kl_loss}

        self.encoder = Model(inputs, [z_mean, z_log_var, z], name="encoder")
        self.decoder = decoder
        self.vae = VAE(self.encoder, self.decoder, self.beta)
        self.vae.compile(optimizer=Adam(learning_rate=0.001))

class EncodedVaR:
    def __init__(self, window_size=90, test_size=10, step_size=21, latent_dim=32, confidence_level=0.95, beta=1.0):
        self.window_size = window_size
        self.test_size = test_size
        self.step_size = step_size
        self.latent_dim = latent_dim
        self.confidence_level = confidence_level
        self.beta = beta
        self.vae = None

    def train_and_predict(self, data, epochs=30, batch_size=32):
        returns = data.pct_change().dropna()
        violations = []
        total_violations = 0
        num_windows = 0

        for start in range(0, len(returns) - self.window_size - self.test_size, self.step_size):
            train_data = returns.iloc[start : start + self.window_size].values.reshape(1, -1)
            test_data = returns.iloc[start + self.window_size : start + self.window_size + self.test_size].values

            if self.vae is None:
                self.vae = VariationalAutoencoder(input_dim=self.window_size, latent_dim=self.latent_dim, beta=self.beta).vae

            self.vae.fit(train_data, epochs=epochs, batch_size=batch_size, verbose=0)

            z_mean, z_log_var, z = self.vae.encoder.predict(train_data)
            predicted = self.vae.decoder.predict(z)[0, -self.test_size:]
            var_threshold = np.percentile(predicted, (1 - self.confidence_level) * 100)

            window_violations = sum(test_data < var_threshold)
            total_violations += window_violations
            violations.append(window_violations)
            num_windows += 1

        overall_violation_percent = (total_violations / (num_windows * self.test_size)) * 100
        avg_violation_percent = np.mean([(v / self.test_size) * 100 for v in violations])
        mean_violations_per_window = np.mean(violations)

        print(f"Overall Violation Percentage: {overall_violation_percent:.2f}%")
        print(f"Average Violation Percentage per Window: {avg_violation_percent:.2f}%")
        print(f"Mean Number of Violations per Window: {mean_violations_per_window:.2f}")

        return violations

def main():
    # Was /content/2y_Nifty50_data.xlsx with the date used as the index.
    # Nifty50_2yr.csv stores Dates as a column, so select NIFTY Index explicitly.
    file_path = find_nifty_file("Nifty50_2yr.csv")
    data = pd.read_csv(file_path)["NIFTY Index"]

    encoded_var = EncodedVaR(window_size=90, test_size=10, step_size=21, latent_dim=32, confidence_level=0.95, beta=1.0)
    encoded_var.train_and_predict(data, epochs=30, batch_size=32)

if __name__ == "__main__":
    main()


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━